# Notebook 05: Robustness Analysis & Sensor Error Sensitivity

## 1. Objectives & Methodological Framing
This notebook evaluates the operational robustness of the frozen Group C Initial-SoC estimator (12 summary features) against:
1. **Operating Point & SoC Distribution:** Characterizing estimation error across the state-of-charge spectrum on clean validation (-10°C) and held-out test (-20°C) data.
2. **Synthetic Measurement Noise & Sensor Bias:** Evaluating sensitivity to controlled voltage and current perturbations injected into the raw measurement sequences prior to summary-feature extraction.
3. **Feature Propagation:** Quantifying how sensor bias and noise propagate mathematically and empirically into derived summary statistics.

**Methodological Guardrails:**
- **Perturbation Labeling:** Voltage and current perturbation levels are **synthetic measurement-noise and sensor-bias stress levels**. They represent controlled sensitivity scenarios, not experimentally measured sensor specifications.
- **Frozen Model & Scaler:** The model (`results/models/ablation_group_c_seed_42.pt`) and scaler (`data/ablation_scaler_group_c.pkl`) remain **completely frozen**. No retraining or fine-tuning under perturbed conditions is performed.
- **Test Set Discipline:** All sensitivity sweeps, bias/noise curves, and vulnerability rankings are conducted **strictly on the Validation set (-10°C)**. The held-out Test set (-20°C) is evaluated strictly post-hoc for 3 predefined scenarios and is not used to tune or select anything.

In [ ]:
import os
import sys
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Selected compute device: {device}")
print(f"PyTorch Version:         {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Runtime Version:    {torch.version.cuda}")
    print(f"GPU Hardware:            {torch.cuda.get_device_name(0)}")

# Reproducibility seed for noise generation
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 2. Load Frozen Model, Scalers, and 50s Datasets
We load the primary frozen Group C model (`ablation_group_c_seed_42.pt`) and its corresponding training-only scaler (`ablation_scaler_group_c.pkl`).

In [ ]:
PROJECT_ROOT = Path(r"c:\Users\sheok\OneDrive\Desktop\New folder (2)\Major Project")
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"
MODELS_DIR   = RESULTS_DIR / "models"
PLOTS_DIR    = RESULTS_DIR / "plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Model Architecture (Group C: 12 Inputs)
class SummaryMLP(nn.Module):
    def __init__(self, in_dim=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

# Load Frozen Model
MODEL_PATH = MODELS_DIR / "ablation_group_c_seed_42.pt"
model = SummaryMLP(in_dim=12).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"Loaded frozen Group C MLP checkpoint from: {MODEL_PATH.name}")

# Load Frozen Group C Scaler (Fitted on training data only)
SCALER_PATH = DATA_DIR / "ablation_scaler_group_c.pkl"
with open(SCALER_PATH, "rb") as f:
    group_c_scaler = pickle.load(f)
print(f"Loaded frozen Group C scaler from: {SCALER_PATH.name}")

# Load Raw Sequence Normalization Scaler
RAW_SCALER_PATH = DATA_DIR / "initial_soc_scaler.npz"
raw_scaler_data = np.load(RAW_SCALER_PATH)
feat_min   = raw_scaler_data["feat_min"]
feat_range = raw_scaler_data["feat_range"]

# Load 50s Nominal Datasets
DATASET_PATH = DATA_DIR / "initial_soc_datasets.pkl"
with open(DATASET_PATH, "rb") as f:
    datasets = pickle.load(f)

dur = 50
X_val_seq = datasets[dur]["X_val"]   # (1045, 500, 3) normalized [0, 1]
y_val     = datasets[dur]["y_val"]   # (1045,) normalized [0, 1]

X_test_seq = datasets[dur]["X_test"] # (871, 500, 3)
y_test     = datasets[dur]["y_test"] # (871,)

print(f"Loaded 50s Nominal Sequences: Validation N={len(X_val_seq)}, Test N={len(X_test_seq)}")


## 3. Clean Performance & SoC Bin Diagnostics

Before applying synthetic perturbations, we examine the baseline error profile across the SoC range on clean validation (-10°C) and clean held-out test (-20°C) data.

**Caveats for -20°C Test Set:**
- Operating range is limited to ~40–100% SoC due to the Panasonic sub-zero early-termination protocol.
- Extreme cold temperature is confounded with sequential cyclic cell aging.

In [ ]:
def extract_group_c_features(X_seq_norm):
    """
    Extracts 12 Group C summary features from raw sequences:
    [V0, I0, T0, Vend, Iend, Tend, dV, dI, dT, meanV, meanI, stdV]
    """
    V_0 = X_seq_norm[:, 0, 0]
    I_0 = X_seq_norm[:, 0, 1]
    T_0 = X_seq_norm[:, 0, 2]
    
    V_end = X_seq_norm[:, -1, 0]
    I_end = X_seq_norm[:, -1, 1]
    T_end = X_seq_norm[:, -1, 2]
    
    dV = V_end - V_0
    dI = I_end - I_0
    dT = T_end - T_0
    
    meanV = np.mean(X_seq_norm[:, :, 0], axis=1)
    meanI = np.mean(X_seq_norm[:, :, 1], axis=1)
    stdV  = np.std(X_seq_norm[:, :, 0], axis=1)
    
    return np.column_stack([
        V_0, I_0, T_0,
        V_end, I_end, T_end,
        dV, dI, dT,
        meanV, meanI, stdV
    ])

def predict_initial_soc(feats_12, scaler, model, batch_size=64):
    feats_scaled = scaler.transform(feats_12)
    X_tensor = torch.tensor(feats_scaled, dtype=torch.float32)
    loader = torch.utils.data.DataLoader(X_tensor, batch_size=batch_size, shuffle=False)
    
    preds = []
    with torch.no_grad():
        for xb in loader:
            xb = xb.to(device)
            p = model(xb).squeeze().cpu().numpy()
            preds.append(p)
    preds_norm = np.concatenate(preds)
    return preds_norm * 100.0  # Return in physical % SoC

# Evaluate clean Validation
feats_val_clean = extract_group_c_features(X_val_seq)
pred_val_clean  = predict_initial_soc(feats_val_clean, group_c_scaler, model)
y_val_phys      = y_val * 100.0

val_clean_rmse = np.sqrt(mean_squared_error(y_val_phys, pred_val_clean))
val_clean_mae  = mean_absolute_error(y_val_phys, pred_val_clean)
val_clean_max  = np.max(np.abs(y_val_phys - pred_val_clean))
print(f"Clean Validation (-10°C): RMSE = {val_clean_rmse:.2f}% | MAE = {val_clean_mae:.2f}% | MaxAE = {val_clean_max:.2f}%")

# Evaluate clean Test (Post-hoc check)
feats_test_clean = extract_group_c_features(X_test_seq)
pred_test_clean  = predict_initial_soc(feats_test_clean, group_c_scaler, model)
y_test_phys      = y_test * 100.0

test_clean_rmse = np.sqrt(mean_squared_error(y_test_phys, pred_test_clean))
test_clean_mae  = mean_absolute_error(y_test_phys, pred_test_clean)
test_clean_max  = np.max(np.abs(y_test_phys - pred_test_clean))
print(f"Clean Test (-20°C Post-Hoc): RMSE = {test_clean_rmse:.2f}% | MAE = {test_clean_mae:.2f}% | MaxAE = {test_clean_max:.2f}%")


### 3.1 Error Breakdown by State-of-Charge (SoC) Bins
We evaluate how estimation errors are distributed across 10% SoC intervals.

In [ ]:
bins = np.linspace(0, 100, 11)
bin_labels = [f"{int(bins[i])}-{int(bins[i+1])}%" for i in range(len(bins)-1)]

bin_records = []
for i in range(len(bins)-1):
    low, high = bins[i], bins[i+1]
    mask_v = (y_val_phys >= low) & (y_val_phys < high)
    mask_t = (y_test_phys >= low) & (y_test_phys < high)
    
    v_rmse = np.sqrt(mean_squared_error(y_val_phys[mask_v], pred_val_clean[mask_v])) if np.sum(mask_v) > 0 else np.nan
    v_mae  = mean_absolute_error(y_val_phys[mask_v], pred_val_clean[mask_v]) if np.sum(mask_v) > 0 else np.nan
    
    t_rmse = np.sqrt(mean_squared_error(y_test_phys[mask_t], pred_test_clean[mask_t])) if np.sum(mask_t) > 0 else np.nan
    t_mae  = mean_absolute_error(y_test_phys[mask_t], pred_test_clean[mask_t]) if np.sum(mask_t) > 0 else np.nan
    
    bin_records.append({
        "SoC_Bin": bin_labels[i],
        "Val_Count": int(np.sum(mask_v)),
        "Val_RMSE": v_rmse,
        "Val_MAE": v_mae,
        "Test_Count": int(np.sum(mask_t)),
        "Test_RMSE": t_rmse,
        "Test_MAE": t_mae
    })

df_bins = pd.DataFrame(bin_records)
df_bins.to_csv(RESULTS_DIR / "robustness_soc_bin_analysis.csv", index=False)
print("=== ERROR BREAKDOWN BY 10% SOC BINS ===")
print(df_bins.to_string(index=False))


In [ ]:
# Plot 1: Parity Plot on Clean Validation
plt.figure(figsize=(7, 6))
plt.scatter(y_val_phys, pred_val_clean, alpha=0.3, s=15, c='navy', label='Validation (-10°C)')
plt.plot([0, 100], [0, 100], 'r--', lw=2, label='Ideal Parity (y=x)')
plt.title('Initial SoC: Predicted vs. Reference (Clean Validation -10°C)')
plt.xlabel('Reference SoC (%)')
plt.ylabel('Estimated Initial SoC (%)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_pred_vs_ref_validation.png", dpi=300)
plt.close()

# Plot 2: Residual Error vs Reference SoC
errors_val = pred_val_clean - y_val_phys
plt.figure(figsize=(8, 5))
plt.scatter(y_val_phys, errors_val, alpha=0.3, s=15, c='teal')
plt.axhline(0, color='red', linestyle='--', lw=1.5)
plt.axhline(5, color='gray', linestyle=':', lw=1)
plt.axhline(-5, color='gray', linestyle=':', lw=1)
plt.title('Estimation Residuals (Predicted - Reference) vs. Reference SoC')
plt.xlabel('Reference SoC (%)')
plt.ylabel('Error (% SoC)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_error_vs_soc.png", dpi=300)
plt.close()

# Plot 3: Error by SoC Bin
valid_bins = df_bins[df_bins['Val_Count'] > 0]
plt.figure(figsize=(9, 5))
bar_w = 0.35
x_pos = np.arange(len(valid_bins))
plt.bar(x_pos - bar_w/2, valid_bins['Val_RMSE'], width=bar_w, label='Val RMSE (-10°C)', color='royalblue')
plt.bar(x_pos + bar_w/2, valid_bins['Val_MAE'], width=bar_w, label='Val MAE (-10°C)', color='lightsteelblue')
plt.xticks(x_pos, valid_bins['SoC_Bin'])
plt.title('Estimation Error across State-of-Charge Intervals (Validation -10°C)')
plt.xlabel('State-of-Charge Interval')
plt.ylabel('Error (% SoC)')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_error_by_soc_bin.png", dpi=300)
plt.close()
print("Diagnostic plots saved to results/plots/")


## 4. Synthetic Measurement-Noise & Sensor-Bias Stress Testing

We evaluate estimator sensitivity by injecting synthetic perturbations directly into the physical unscaled observation sequences before extracting summary features.

**Controlled Perturbation Scenarios:**
- **Voltage Bias:** $b_V \in \{-20, -10, +10, +20\}\text{ mV}$
- **Current Bias:** $b_I \in \{-100, -50, +50, +100\}\text{ mA}$
- **Voltage Noise:** $\sigma_V \in \{2, 5, 10\}\text{ mV}$ (Zero-mean Gaussian)
- **Current Noise:** $\sigma_I \in \{10, 25, 50\}\text{ mA}$ (Zero-mean Gaussian)
- **Combined Stress Scenarios:** Multiple co-occurring bias and noise offsets.

In [ ]:
def apply_perturbations_and_evaluate(X_raw_norm, y_true_norm, b_V=0.0, b_I=0.0, sigma_V=0.0, sigma_I=0.0, seed=42):
    """
    1. Unscales raw sequence to physical units.
    2. Injects synthetic physical bias (V, A) and Gaussian noise (V, A).
    3. Re-normalizes channels to [0, 1] using training min/range.
    4. Extracts Group C 12 summary features.
    5. Applies frozen train-only scaler and runs frozen model.
    """
    rng = np.random.RandomState(seed)
    N, W, C = X_raw_norm.shape
    
    # 1. Physical unscaling
    V_phys = X_raw_norm[:, :, 0] * feat_range[0] + feat_min[0]
    I_phys = X_raw_norm[:, :, 1] * feat_range[1] + feat_min[1]
    T_phys = X_raw_norm[:, :, 2] * feat_range[2] + feat_min[2]
    
    # 2. Add physical bias and noise
    if sigma_V > 0.0:
        V_phys = V_phys + b_V + rng.normal(0.0, sigma_V, size=(N, W))
    else:
        V_phys = V_phys + b_V
        
    if sigma_I > 0.0:
        I_phys = I_phys + b_I + rng.normal(0.0, sigma_I, size=(N, W))
    else:
        I_phys = I_phys + b_I
        
    # 3. Re-scale channels back to [0, 1]
    X_pert = np.zeros_like(X_raw_norm)
    X_pert[:, :, 0] = (V_phys - feat_min[0]) / feat_range[0]
    X_pert[:, :, 1] = (I_phys - feat_min[1]) / feat_range[1]
    X_pert[:, :, 2] = (T_phys - feat_min[2]) / feat_range[2]
    
    # 4. Extract 12 Group C summary features
    feats_12 = extract_group_c_features(X_pert)
    
    # 5. Predict with frozen scaler and model
    preds_phys = predict_initial_soc(feats_12, group_c_scaler, model)
    y_phys     = y_true_norm * 100.0
    
    rmse = np.sqrt(mean_squared_error(y_phys, preds_phys))
    mae  = mean_absolute_error(y_phys, preds_phys)
    max_ae = np.max(np.abs(y_phys - preds_phys))
    
    return rmse, mae, max_ae


### 4.1 Systematic Validation Sensitivity Sweeps
We evaluate individual bias sweeps, noise sweeps, and combined stress conditions on the Validation (-10°C) split.

In [ ]:
scenarios = []

# A. Clean Baseline
rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val)
scenarios.append({"Suite": "Baseline", "Scenario": "Clean", "b_V_mV": 0.0, "b_I_mA": 0.0, "sigma_V_mV": 0.0, "sigma_I_mA": 0.0, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

# B. Voltage Bias Sweep
for b_v in [-20.0, -10.0, 10.0, 20.0]:
    rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val, b_V=b_v/1000.0)
    scenarios.append({"Suite": "Voltage Bias", "Scenario": f"b_V={b_v:+.0f}mV", "b_V_mV": b_v, "b_I_mA": 0.0, "sigma_V_mV": 0.0, "sigma_I_mA": 0.0, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

# C. Current Bias Sweep
for b_i in [-100.0, -50.0, 50.0, 100.0]:
    rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val, b_I=b_i/1000.0)
    scenarios.append({"Suite": "Current Bias", "Scenario": f"b_I={b_i:+.0f}mA", "b_V_mV": 0.0, "b_I_mA": b_i, "sigma_V_mV": 0.0, "sigma_I_mA": 0.0, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

# D. Voltage Noise Sweep
for s_v in [2.0, 5.0, 10.0]:
    rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val, sigma_V=s_v/1000.0)
    scenarios.append({"Suite": "Voltage Noise", "Scenario": f"sigma_V={s_v:.0f}mV", "b_V_mV": 0.0, "b_I_mA": 0.0, "sigma_V_mV": s_v, "sigma_I_mA": 0.0, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

# E. Current Noise Sweep
for s_i in [10.0, 25.0, 50.0]:
    rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val, sigma_I=s_i/1000.0)
    scenarios.append({"Suite": "Current Noise", "Scenario": f"sigma_I={s_i:.0f}mA", "b_V_mV": 0.0, "b_I_mA": 0.0, "sigma_V_mV": 0.0, "sigma_I_mA": s_i, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

# F. Combined Bias Permutations
for b_v in [-10.0, 10.0]:
    for b_i in [-50.0, 50.0]:
        rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val, b_V=b_v/1000.0, b_I=b_i/1000.0)
        scenarios.append({"Suite": "Combined Bias", "Scenario": f"b_V={b_v:+.0f}mV, b_I={b_i:+.0f}mA", "b_V_mV": b_v, "b_I_mA": b_i, "sigma_V_mV": 0.0, "sigma_I_mA": 0.0, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

# G. Combined Noise
rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val, sigma_V=5.0/1000.0, sigma_I=25.0/1000.0)
scenarios.append({"Suite": "Combined Noise", "Scenario": "sigma_V=5mV, sigma_I=25mA", "b_V_mV": 0.0, "b_I_mA": 0.0, "sigma_V_mV": 5.0, "sigma_I_mA": 25.0, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

# H. Severe Combined Bias + Noise
rmse, mae, max_ae = apply_perturbations_and_evaluate(X_val_seq, y_val, b_V=10.0/1000.0, b_I=50.0/1000.0, sigma_V=5.0/1000.0, sigma_I=25.0/1000.0)
scenarios.append({"Suite": "Combined Bias+Noise", "Scenario": "b_V=+10mV, b_I=+50mA, sig_V=5mV, sig_I=25mA", "b_V_mV": 10.0, "b_I_mA": 50.0, "sigma_V_mV": 5.0, "sigma_I_mA": 25.0, "Val_RMSE": rmse, "Val_MAE": mae, "Val_MaxAE": max_ae})

df_scenarios = pd.DataFrame(scenarios)
df_scenarios.to_csv(RESULTS_DIR / "robustness_validation_summary.csv", index=False)
print("=== VALIDATION PERTURBATION SUMMARY TABLE ===")
print(df_scenarios[["Suite", "Scenario", "Val_RMSE", "Val_MAE", "Val_MaxAE"]].to_string(index=False))


In [ ]:
# Plot 4: Sensor Bias Sensitivity Curve
df_vb = df_scenarios[df_scenarios['Suite'] == 'Voltage Bias'].copy()
clean_row = df_scenarios[df_scenarios['Scenario'] == 'Clean'].iloc[0]
df_vb = pd.concat([df_vb, pd.DataFrame([clean_row])]).sort_values('b_V_mV')

df_ib = df_scenarios[df_scenarios['Suite'] == 'Current Bias'].copy()
df_ib = pd.concat([df_ib, pd.DataFrame([clean_row])]).sort_values('b_I_mA')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(df_vb['b_V_mV'], df_vb['Val_RMSE'], 'o-', lw=2, color='crimson')
ax1.axhline(clean_row['Val_RMSE'], color='gray', linestyle='--', alpha=0.7, label='Clean Baseline')
ax1.set_title('Sensitivity to Voltage Sensor Bias (b_V)')
ax1.set_xlabel('Voltage Bias Offset (mV)')
ax1.set_ylabel('Validation RMSE (% SoC)')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(df_ib['b_I_mA'], df_ib['Val_RMSE'], 's-', lw=2, color='darkorange')
ax2.axhline(clean_row['Val_RMSE'], color='gray', linestyle='--', alpha=0.7, label='Clean Baseline')
ax2.set_title('Sensitivity to Current Sensor Bias (b_I)')
ax2.set_xlabel('Current Bias Offset (mA)')
ax2.set_ylabel('Validation RMSE (% SoC)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_sensor_bias_sensitivity.png", dpi=300)
plt.close()

# Plot 5: Sensor Noise Sensitivity Curve
df_vn = df_scenarios[df_scenarios['Suite'] == 'Voltage Noise'].copy()
df_vn = pd.concat([df_vn, pd.DataFrame([clean_row])]).sort_values('sigma_V_mV')

df_in = df_scenarios[df_scenarios['Suite'] == 'Current Noise'].copy()
df_in = pd.concat([df_in, pd.DataFrame([clean_row])]).sort_values('sigma_I_mA')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(df_vn['sigma_V_mV'], df_vn['Val_RMSE'], 'o-', lw=2, color='darkblue')
ax1.axhline(clean_row['Val_RMSE'], color='gray', linestyle='--', alpha=0.7, label='Clean Baseline')
ax1.set_title('Sensitivity to Voltage Measurement Noise (sigma_V)')
ax1.set_xlabel('Gaussian Noise Std Dev (mV)')
ax1.set_ylabel('Validation RMSE (% SoC)')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(df_in['sigma_I_mA'], df_in['Val_RMSE'], 's-', lw=2, color='darkgreen')
ax2.axhline(clean_row['Val_RMSE'], color='gray', linestyle='--', alpha=0.7, label='Clean Baseline')
ax2.set_title('Sensitivity to Current Measurement Noise (sigma_I)')
ax2.set_xlabel('Gaussian Noise Std Dev (mA)')
ax2.set_ylabel('Validation RMSE (% SoC)')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_sensor_noise_sensitivity.png", dpi=300)
plt.close()
print("Sensitivity plots saved to results/plots/")


## 5. Post-Hoc Test Set Verification (-20°C Held-Out Split)

We perform a strictly limited post-hoc evaluation on the held-out -20°C test data across 3 predefined conditions:
1. **Clean Test Baseline**
2. **Moderate Combined Noise:** $\sigma_V = 5\text{ mV}, \sigma_I = 25\text{ mA}$
3. **Moderate Combined Bias:** $b_V = +10\text{ mV}, b_I = +50\text{ mA}$

**Strict Policy:** These test results are purely post-hoc and do not influence any model, feature, or threshold selection.

In [ ]:
test_checks = []

# 1. Clean Test
t_rmse, t_mae, t_max = apply_perturbations_and_evaluate(X_test_seq, y_test)
test_checks.append({"Condition": "1. Clean Baseline", "Test_RMSE": t_rmse, "Test_MAE": t_mae, "Test_MaxAE": t_max})

# 2. Moderate Combined Noise
t_rmse, t_mae, t_max = apply_perturbations_and_evaluate(X_test_seq, y_test, sigma_V=5.0/1000.0, sigma_I=25.0/1000.0)
test_checks.append({"Condition": "2. Moderate Noise (5mV, 25mA)", "Test_RMSE": t_rmse, "Test_MAE": t_mae, "Test_MaxAE": t_max})

# 3. Moderate Combined Bias
t_rmse, t_mae, t_max = apply_perturbations_and_evaluate(X_test_seq, y_test, b_V=10.0/1000.0, b_I=50.0/1000.0)
test_checks.append({"Condition": "3. Moderate Bias (+10mV, +50mA)", "Test_RMSE": t_rmse, "Test_MAE": t_mae, "Test_MaxAE": t_max})

df_test_checks = pd.DataFrame(test_checks)
print("=== POST-HOC TEST SET CHECKS (-20°C) ===")
print(df_test_checks.to_string(index=False))

# Combined comparison table
clean_vs_pert = pd.DataFrame([
    {"Split": "Validation (-10°C)", "Condition": "Clean", "RMSE": val_clean_rmse, "MAE": val_clean_mae},
    {"Split": "Validation (-10°C)", "Condition": "Moderate Noise (5mV, 25mA)", "RMSE": df_scenarios[df_scenarios['Scenario'] == 'sigma_V=5mV, sigma_I=25mA']['Val_RMSE'].values[0], "MAE": df_scenarios[df_scenarios['Scenario'] == 'sigma_V=5mV, sigma_I=25mA']['Val_MAE'].values[0]},
    {"Split": "Validation (-10°C)", "Condition": "Moderate Bias (+10mV, +50mA)", "RMSE": df_scenarios[df_scenarios['Scenario'] == 'b_V=+10mV, b_I=+50mA']['Val_RMSE'].values[0], "MAE": df_scenarios[df_scenarios['Scenario'] == 'b_V=+10mV, b_I=+50mA']['Val_MAE'].values[0]},
    {"Split": "Test (-20°C Post-Hoc)", "Condition": "Clean", "RMSE": test_checks[0]['Test_RMSE'], "MAE": test_checks[0]['Test_MAE']},
    {"Split": "Test (-20°C Post-Hoc)", "Condition": "Moderate Noise (5mV, 25mA)", "RMSE": test_checks[1]['Test_RMSE'], "MAE": test_checks[1]['Test_MAE']},
    {"Split": "Test (-20°C Post-Hoc)", "Condition": "Moderate Bias (+10mV, +50mA)", "RMSE": test_checks[2]['Test_RMSE'], "MAE": test_checks[2]['Test_MAE']}
])
clean_vs_pert.to_csv(RESULTS_DIR / "robustness_clean_vs_perturbed.csv", index=False)
print("\n=== CLEAN VS. PERTURBED COMPARISON TABLE ===")
print(clean_vs_pert.to_string(index=False))
